# 一 构造感知器与神经网络

**教学说明（本科生）：**

本节从零开始手动实现神经网络，深入理解神经网络的工作原理。

**学习内容：**
1. **单神经元模型**：感知器的基本形式
2. **激活函数**：非线性变换的重要性
3. **梯度下降**：神经网络的训练算法
4. **多层网络**：引入隐藏层解决非线性问题

**核心概念：**
- **线性组合**：z = Xw + b
- **激活函数**：引入非线性，如Sigmoid、ReLU
- **损失函数**：衡量预测与真实的差异
- **反向传播**：计算梯度并更新参数

**前置知识：**
- 微积分：链式法则、偏导数
- 线性代数：矩阵乘法
- 概率统计：损失函数的意义

**学习目标：**
理解神经网络的数学原理，掌握从零实现神经网络的能力。

---
### 学习重点

- 理解单神经元模型的基本形式：z = Xw + b
- 掌握不同激活函数的特性和用途
- 理解前向传播和反向传播的数学原理
- 体会隐藏层如何赋予网络非线性表达能力

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

## 构建线性可分数据集

**教学说明（本科生）：**

本节生成用于神经网络训练的线性可分数据集。

**make_blobs数据集：**
- 生成多个高斯分布的簇
- 类别之间线性可分
- 适合演示神经网络的基本功能

**学习重点：**
理解线性可分数据的特点，体会单个神经元在此类数据上的表现。

In [ ]:
np.random.seed(42)

X, y = make_blobs(
    n_samples=300,
    centers=2,
    n_features=2,
    cluster_std=1.8,
    random_state=42
)

# Convert labels to 0 and 1
y = y.reshape(-1, 1)

plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), alpha=0.8)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Linearly separable dataset")
plt.show()

## 划分数据集

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 设置激活函数

**教学说明（本科生）：**

本节定义和比较不同的激活函数。

**常见的激活函数：**
1. **Sigmoid**：σ(z) = 1 / (1 + e^-z)
   - 输出范围：(0, 1)
   - 适合输出概率
   - 问题：容易饱和，梯度消失

2. **ReLU**（Rectified Linear Unit）：ReLU(z) = max(0, z)
   - 输出范围：[0, ∞)
   - 计算简单
   - 缓解梯度消失问题

3. **无激活**：f(z) = z
   - 线性变换

**激活函数的作用：**
- 引入非线性，使网络能拟合非线性函数
- 增加模型的表达能力

**学习重点：**
理解激活函数的必要性，掌握不同激活函数的特点。

### 激活函数对比

| 函数 | 公式 | 输出范围 | 优点 | 缺点 | 适用场景 |
|------|------|---------|------|------|---------|
| Sigmoid | 1/(1+e⁻ᶻ) | (0, 1) | 可解释为概率 | 梯度消失 | 二分类输出层 |
| ReLU | max(0, z) | [0, ∞) | 计算快，缓解梯度消失 | 神经元死亡 | 隐藏层 🔥 |
| Tanh | (eᶻ-e⁻ᶻ)/(eᶻ+e⁻ᶻ) | (-1, 1) | 以0为中心 | 梯度消失 | 隐藏层 |
| 线性 | z | (-∞, ∞) | 简单 | 无法引入非线性 | 回归输出层 |

> 💡 **教学提示**：激活函数是神经网络的"灵魂"——如果没有它们，无论多少层网络都等价于一个线性变换，无法解决非线性问题。

### 如何解读激活函数比较结果

观察输出表格中的 `test_accuracy`：
- **relu**通常表现最好，因为它缓解了梯度消失问题
- **logistic** (sigmoid) 在深层网络中可能收敛较慢
- **tanh** 以0为中心的特性在某些任务上有优势

实际应用中，**ReLU** 是隐藏层的默认首选。

In [ ]:
def step_function(z):
    return (z >= 0).astype(float)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

z = np.linspace(-8, 8, 400)

plt.figure(figsize=(10, 6))
plt.plot(z, z, label="No activation: f(z)=z")
plt.plot(z, step_function(z), label="Step activation")
plt.plot(z, sigmoid(z), label="Sigmoid activation")
plt.plot(z, relu(z), label="ReLU activation")
plt.xlabel("z")
plt.ylabel("f(z)")
plt.title("Activation functions")
plt.legend()
plt.show()

## 单神经元无激活

**教学说明（本科生）：**

本节实现没有激活函数的单神经元模型。

**模型形式：**
```
y = z = Xw + b
```

**训练方法：**
- 损失函数：MSE（均方误差）
- 优化算法：梯度下降
- 更新公式：
  - w = w - lr * ∂loss/∂w
  - b = b - lr * ∂loss/∂b

**学习重点：**
理解最简单的神经网络训练过程，掌握梯度下降的实现细节。

In [ ]:
# Initialize parameters
w_no_act = np.zeros((X_train.shape[1], 1))
b_no_act = 0.0

learning_rate = 0.05
epochs = 200

loss_history_no_act = []

def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

n = X_train.shape[0]

for epoch in range(epochs):
    # Forward
    z = X_train @ w_no_act + b_no_act
    y_pred = z
    
    # Loss
    loss = mse_loss(y_train, y_pred)
    loss_history_no_act.append(loss)
    
    # Gradients
    dw = (2 / n) * X_train.T @ (y_pred - y_train)
    db = (2 / n) * np.sum(y_pred - y_train)
    
    # Update
    w_no_act -= learning_rate * dw
    b_no_act -= learning_rate * db

    if epoch % 1 == 0 or epoch == epochs - 1:
        print(f"[No activation] Epoch {epoch+1:03d}/{epochs} | Loss = {loss:.6f}")

## 评估未激活的神经元

**教学说明（本科生）：**

本节评估没有激活函数的神经元的分类性能。

**阈值分类：**
- 由于没有激活函数，输出是连续值
- 使用0.5作为阈值进行分类
- z >= 0.5 → 预测为1
- z < 0.5 → 预测为0

**学习重点：**
理解输出阈值在分类中的作用，体会无激活函数模型的局限性。

In [ ]:
z_train_no_act = X_train @ w_no_act + b_no_act
z_test_no_act = X_test @ w_no_act + b_no_act

y_train_pred_no_act = (z_train_no_act >= 0.5).astype(int)
y_test_pred_no_act = (z_test_no_act >= 0.5).astype(int)

print("No activation model")
print("Train accuracy:", accuracy_score(y_train, y_train_pred_no_act))
print("Test accuracy:", accuracy_score(y_test, y_test_pred_no_act))

## 未激活损失可视化

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(loss_history_no_act, linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Loss without activation")
plt.show()

## 单个神经元加入sigmoid激活函数

**教学说明（本科生）：**

本节实现带有Sigmoid激活函数的单神经元模型。

**模型形式：**
```
z = Xw + b
y = σ(z) = 1 / (1 + e^-z)
```

**损失函数：**
- 二元交叉熵（Binary Cross-Entropy）
- BCE = -[y*log(y_pred) + (1-y)*log(1-y_pred)]

**梯度计算：**
- ∂BCE/∂w = (y_pred - y) * x
- ∂BCE/∂b = (y_pred - y)

**Sigmoid的优势：**
- 输出范围(0, 1)，可解释为概率
- 导数形式简单：σ'(z) = σ(z) * (1 - σ(z))

**学习重点：**
理解Sigmoid激活函数与BCE损失函数的配合，掌握逻辑回归的实现。

### Sigmoid + BCE 配合原理

Sigmoid的输出在(0,1)之间，可以解释为概率。
二元交叉熵(BCE)损失对概率输出的梯度计算简单：

```
∂BCE/∂w = (y_pred - y_true) × x
∂BCE/∂b = (y_pred - y_true)
```

**巧合？** 不是巧合！Sigmoid + BCE的梯度形式恰好与线性回归 + MSE相同——这就是所谓的"广义线性模型"的统一性。

In [ ]:
# Initialize parameters
w_sigmoid = np.zeros((X_train.shape[1], 1))
b_sigmoid = 0.0

learning_rate = 0.1
epochs = 500

loss_history_sigmoid = []

def binary_cross_entropy(y_true, y_pred):
    eps = 1e-8
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

n = X_train.shape[0]

for epoch in range(epochs):
    # Forward
    z = X_train @ w_sigmoid + b_sigmoid
    y_pred = sigmoid(z)
    
    # Loss
    loss = binary_cross_entropy(y_train, y_pred)
    loss_history_sigmoid.append(loss)
    
    # Gradients for sigmoid + BCE
    dz = y_pred - y_train
    dw = (1 / n) * X_train.T @ dz
    db = (1 / n) * np.sum(dz)
    
    # Update
    w_sigmoid -= learning_rate * dw
    b_sigmoid -= learning_rate * db

    if epoch % 1 == 0 or epoch == epochs - 1:
        print(f"[Sigmoid] Epoch {epoch+1:03d}/{epochs} | Loss = {loss:.6f}")

## 评估有激活函数的神经元

In [ ]:
y_train_prob_sigmoid = sigmoid(X_train @ w_sigmoid + b_sigmoid)
y_test_prob_sigmoid = sigmoid(X_test @ w_sigmoid + b_sigmoid)

y_train_pred_sigmoid = (y_train_prob_sigmoid >= 0.5).astype(int)
y_test_pred_sigmoid = (y_test_prob_sigmoid >= 0.5).astype(int)

print("Sigmoid neuron")
print("Train accuracy:", accuracy_score(y_train, y_train_pred_sigmoid))
print("Test accuracy:", accuracy_score(y_test, y_test_pred_sigmoid))

## 可视化loss曲线

**教学说明（本科生）：**

本节可视化Sigmoid神经元训练过程中的损失变化。

**学习重点：**
- 观察损失是否随训练下降
- 理解损失曲线反映的训练动态
- 对比无激活函数的损失曲线

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(loss_history_sigmoid, linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Loss with sigmoid activation")
plt.show()

## 对比loss曲线

**教学说明（本科生）：**

本节对比有无激活函数的损失曲线。

**观察要点：**
1. Sigmoid + BCE 的损失下降是否更稳定
2. 训练最终达到的损失水平
3. 收敛速度的差异

**学习重点：**
理解激活函数和损失函数的选择对训练过程的影响。

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(loss_history_no_act, label="No activation + MSE")
plt.plot(loss_history_sigmoid, label="Sigmoid + BCE")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss comparison")
plt.legend()
plt.show()

## 可视化决策边界

**教学说明（本科生）：**

本节定义决策边界可视化函数。

**决策边界的意义：**
- 显示神经网络在特征空间中的分类决策
- 线性决策边界：单个神经元只能学习线性分类
- 非线性决策边界：需要隐藏层和激活函数

**可视化方法：**
1. 在特征空间网格上预测每个点的类别
2. 用不同颜色表示不同类别
3. 边界即为决策边界

**学习重点：**
理解决策边界的几何意义，体会激活函数对决策能力的影响。

In [ ]:
def plot_decision_boundary_linear(X_data, y_data, w, b, title, use_sigmoid=False):
    x_min, x_max = X_data[:, 0].min() - 1, X_data[:, 0].max() + 1
    y_min, y_max = X_data[:, 1].min() - 1, X_data[:, 1].max() + 1
    
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )
    
    grid = np.c_[xx.ravel(), yy.ravel()]
    z = grid @ w + b
    
    if use_sigmoid:
        probs = sigmoid(z)
        Z = (probs >= 0.5).astype(int)
    else:
        Z = (z >= 0.5).astype(int)
    
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, alpha=0.3)
    plt.scatter(X_data[:, 0], X_data[:, 1], c=y_data.ravel(), edgecolor="k", alpha=0.8)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)
    plt.show()

plot_decision_boundary_linear(
    X_train, y_train, w_no_act, b_no_act,
    title="Decision boundary: no activation",
    use_sigmoid=False
)

plot_decision_boundary_linear(
    X_train, y_train, w_sigmoid, b_sigmoid,
    title="Decision boundary: sigmoid activation",
    use_sigmoid=True
)

## 为什么我们需要神经网络？

**教学说明（本科生）：**

本节引出神经网络的必要性。

**单个感知器的局限：**
- 只能学习线性决策边界
- 对于线性不可分数据（如异或问题、月牙形数据），无能为力

**解决方案：**
1. **添加隐藏层**：引入非线性变换
2. **添加激活函数**：增强模型表达能力
3. **多层前馈网络**：通过组合多个线性变换和非线性激活

**学习重点：**
理解神经网络发展的动机，体会从简单到复杂的建模思路。

### 单层感知器的局限

| 问题类型 | 单层感知器 | 多层神经网络 |
|---------|-----------|-------------|
| 线性可分（如AND/OR） | ✅ 可以 | ✅ 可以 |
| 线性不可分（如XOR） | ❌ 不行 | ✅ 可以 |
| 月牙形数据 | ❌ 不行 | ✅ 可以 |

> 💡 **教学提示**：1969年Minsky在《感知器》一书中证明了单层感知器无法解决XOR问题，这导致AI进入第一次寒冬。而多层网络+激活函数正是突破这一限制的关键！

## 生成非线性数据集

**教学说明（本科生）：**

本节生成非线性可分的数据集，用于演示神经网络的优势。

**make_moons数据集：**
- 两个交错的月牙形类别
- 线性不可分
- 是测试非线性分类算法的经典数据集

**学习重点：**
理解非线性可分数据的特点，体会神经网络在此类数据上的优势。

In [ ]:
X2, y2 = make_moons(n_samples=400, noise=0.2, random_state=42)
y2 = y2.reshape(-1, 1)

plt.figure(figsize=(8, 6))
plt.scatter(X2[:, 0], X2[:, 1], c=y2.ravel(), alpha=0.8)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Nonlinear dataset")
plt.show()

## 划分数据集

**教学说明（本科生）：**

本节划分训练集和测试集。

**学习重点：**
理解模型评估的基本方法，确保评估的客观性。

In [ ]:
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.3, random_state=42, stratify=y2
)

scaler2 = StandardScaler()
X2_train = scaler2.fit_transform(X2_train)
X2_test = scaler2.transform(X2_test)

## 构建一个简单的2层神经网络

**教学说明（本科生）：**

本节实现一个简单的两层神经网络（多层感知机）。

**网络结构：**
```
输入层（2个神经元）
    ↓
隐藏层（8个神经元，ReLU激活）
    ↓
输出层（1个神经元，Sigmoid激活）
```

**前向传播：**
```
Z1 = X @ W1 + b1
A1 = ReLU(Z1)
Z2 = A1 @ W2 + b2
A2 = Sigmoid(Z2)
```

**参数初始化：**
- 权重W：小随机数（如0.1 * np.random.randn）
- 偏置b：初始化为0

**学习重点：**
理解多层神经网络的前向传播过程，掌握参数初始化的方法。

### 前向传播 形状追踪

```
输入: X[100, 2]  (100个样本, 2个特征)
  ↓
Z1 = X@W1 + b1  →  [100, 8]  (8个隐藏神经元)
A1 = ReLU(Z1)    →  [100, 8]  (引入非线性)
  ↓
Z2 = A1@W2 + b2 →  [100, 1]  (1个输出)
A2 = Sigmoid(Z2) →  [100, 1]  (概率输出)
```

> 💡 **教学提示**：跟踪张量形状是调试神经网络最重要的技能之一。形状不匹配是90%以上错误的根源！

In [ ]:
np.random.seed(42)

input_dim = 2
hidden_dim = 8
output_dim = 1

W1 = np.random.randn(input_dim, hidden_dim) * 0.1
b1 = np.zeros((1, hidden_dim))

W2 = np.random.randn(hidden_dim, output_dim) * 0.1
b2 = np.zeros((1, output_dim))

## 用梯度下降法训练神经网络

**教学说明（本科生）：**

本节实现神经网络的训练过程，包括前向传播、反向传播和参数更新。

**反向传播原理：**
1. 前向传播计算每个层的输出
2. 计算损失函数
3. 从输出层向输入层反向传播梯度
4. 使用链式法则计算每个参数的梯度

**梯度计算：**
```
dZ2 = A2 - y
dW2 = A1.T @ dZ2 / n
db2 = sum(dZ2) / n

dA1 = dZ2 @ W2.T
dZ1 = dA1 * ReLU'(Z1)
dW1 = X.T @ dZ1 / n
db1 = sum(dZ1) / n
```

**参数更新：**
```
W = W - learning_rate * dW
b = b - learning_rate * db
```

**学习重点：**
理解反向传播的核心思想，掌握多层神经网络的训练流程。

### 反向传播链式法则

```
损失 L = BCE(y, A2)
  ↓
dZ2 = A2 - y                    (dL/dZ2)
dW2 = A1^T @ dZ2 / n            (dL/dW2)
  ↓
dA1 = dZ2 @ W2^T                (dL/dA1)
dZ1 = dA1 * (Z1 > 0)            ReLU导数
  ↓
dW1 = X^T @ dZ1 / n             (dL/dW1)
```

> 💡 **教学提示**：反向传播本质上是微积分中的**链式法则**。想象一条流水线——从最终产品（损失）开始，一步步往回追溯，计算每个环节对最终质量的影响。

In [ ]:
learning_rate = 0.05
epochs = 3000
n = X2_train.shape[0]

nn_loss_history = []

for epoch in range(epochs):
    # Forward pass
    Z1 = X2_train @ W1 + b1
    A1 = relu(Z1)
    
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)
    
    # Loss
    loss = binary_cross_entropy(y2_train, A2)
    nn_loss_history.append(loss)
    
    # Backward pass
    dZ2 = A2 - y2_train
    dW2 = (1 / n) * A1.T @ dZ2
    db2 = (1 / n) * np.sum(dZ2, axis=0, keepdims=True)
    
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * (Z1 > 0)   # derivative of ReLU
    dW1 = (1 / n) * X2_train.T @ dZ1
    db1 = (1 / n) * np.sum(dZ1, axis=0, keepdims=True)
    
    # Update
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

    if epoch % 300 == 0 or epoch == epochs - 1:
        print(f"[Neural network] Epoch {epoch+1:04d}/{epochs} | Loss = {loss:.6f}")

## 可视化loss曲线

**教学说明（本科生）：**

本节可视化神经网络训练过程中的损失变化。

**学习重点：**
- 观察损失是否随训练下降
- 理解损失曲线反映的训练动态
- 对比单神经元与多层网络的训练效果

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(nn_loss_history, linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Neural network training loss")
plt.show()

## 评价结果

**教学说明（本科生）：**

本节评估神经网络的分类性能。

**学习重点：**
- 对比训练集和测试集性能
- 理解过拟合和欠拟合的现象
- 体会隐藏层对模型性能的提升

In [ ]:
# Train predictions
A1_train = relu(X2_train @ W1 + b1)
A2_train = sigmoid(A1_train @ W2 + b2)
y2_train_pred = (A2_train >= 0.5).astype(int)

# Test predictions
A1_test = relu(X2_test @ W1 + b1)
A2_test = sigmoid(A1_test @ W2 + b2)
y2_test_pred = (A2_test >= 0.5).astype(int)

print("Neural network")
print("Train accuracy:", accuracy_score(y2_train, y2_train_pred))
print("Test accuracy:", accuracy_score(y2_test, y2_test_pred))

## 可视化决策边界

**教学说明（本科生）：**

本节可视化神经网络的决策边界。

**观察要点：**
- 与单神经元的线性决策边界对比
- 神经网络可以学习复杂的非线性决策边界
- 决策边界如何适应数据的形状

**学习重点：**
理解多层神经网络的决策能力，体会隐藏层和激活函数的作用。

In [ ]:
def plot_decision_boundary_nn(X_data, y_data, W1, b1, W2, b2, title):
    x_min, x_max = X_data[:, 0].min() - 1, X_data[:, 0].max() + 1
    y_min, y_max = X_data[:, 1].min() - 1, X_data[:, 1].max() + 1
    
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )
    
    grid = np.c_[xx.ravel(), yy.ravel()]
    
    Z1 = grid @ W1 + b1
    A1 = relu(Z1)
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)
    
    Z = (A2 >= 0.5).astype(int).reshape(xx.shape)
    
    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, alpha=0.3)
    plt.scatter(X_data[:, 0], X_data[:, 1], c=y_data.ravel(), edgecolor="k", alpha=0.8)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)
    plt.show()

plot_decision_boundary_nn(
    X2_train, y2_train, W1, b1, W2, b2,
    title="Decision boundary: simple neural network"
)

## 结果比较

**教学说明（本科生）：**

本节对比不同模型的性能。

**对比模型：**
1. **无激活的单神经元**：线性模型
2. **Sigmoid单神经元**：逻辑回归
3. **两层神经网络**：多层感知机

**学习重点：**
理解模型复杂度与性能的关系，体会深度学习的优势。

In [ ]:
summary = {
    "Model": [
        "Single neuron without activation",
        "Single neuron with sigmoid activation",
        "Two-layer neural network"
    ],
    "Train accuracy": [
        accuracy_score(y_train, y_train_pred_no_act),
        accuracy_score(y_train, y_train_pred_sigmoid),
        accuracy_score(y2_train, y2_train_pred)
    ],
    "Test accuracy": [
        accuracy_score(y_test, y_test_pred_no_act),
        accuracy_score(y_test, y_test_pred_sigmoid),
        accuracy_score(y2_test, y2_test_pred)
    ]
}

for i in range(len(summary["Model"])):
    print(summary["Model"][i])
    print("Train accuracy:", round(summary["Train accuracy"][i], 4))
    print("Test accuracy:", round(summary["Test accuracy"][i], 4))
    print("-" * 40)

# 二 sklearn神经网络实现

**教学说明（本科生）：**

本节介绍使用sklearn的MLPClassifier实现神经网络。

**MLPClassifier特点：**
- 基于神经网络的多层分类器
- 支持多种激活函数：relu、logistic、tanh
- 内置梯度下降优化器
- 自动处理反向传播

**学习内容：**
1. 如何使用sklearn训练神经网络
2. 不同激活函数对性能的影响
3. 网络结构对性能的影响
4. 学习率对训练的影响

**学习重点：**
掌握使用sklearn实现神经网络的基本方法，理解超参数对模型性能的影响。

---
### 学习重点

- 掌握MLPClassifier的使用方法
- 理解不同激活函数对训练的影响
- 理解网络结构（宽度和深度）对性能的影响
- 理解学习率对收敛速度和稳定性的影响

## 基于MLP分类器的人工神经网络

**教学说明（本科生）：**

本节介绍sklearn中的人工神经网络实现。

**核心功能：**
- **真实数据集**：使用乳腺癌 Wisconsin 数据集
- **激活功能**：比较relu、logistic、tanh等激活函数
- **损失函数**：内置的交叉熵损失
- **基于梯度下降的优化**：自动反向传播
- **超参数比较**：激活函数、网络结构、学习率

**学习重点：**
理解sklearn神经网络的使用方法，掌握超参数调优的基本思路。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 加载数据

**教学说明（本科生）：**

本节加载乳腺癌 Wisconsin 数据集。

**数据集信息：**
- 569个样本
- 30个特征（平均值、标准差、最坏值）
- 2个类别：恶性、良性
- 经典的二分类数据集

**学习重点：**
理解乳腺癌数据集的特点，掌握sklearn数据集的加载方法。

In [ ]:
data = load_breast_cancer()

X = data.data
y = data.target

feature_names = data.feature_names
target_names = data.target_names

print("Dataset name: Breast Cancer Wisconsin")
print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
print("Class names:", target_names)

## 检查数据集

In [ ]:
df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

display(df.head())

## 检查类分布情况

In [ ]:
class_counts = pd.Series(y).value_counts().sort_index()
class_df = pd.DataFrame({
    "class_label": class_counts.index,
    "class_name": [target_names[i] for i in class_counts.index],
    "count": class_counts.values
})

display(class_df)

plt.figure(figsize=(6, 4))
plt.bar(class_df["class_name"], class_df["count"])
plt.xlabel("Class")
plt.ylabel("Count")
plt.title("Class distribution")
plt.show()

## 划分数据集

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 特征缩放

**教学说明（本科生）：**

本节对数据进行标准化处理。

**为什么需要标准化：**
- 神经网络对输入数据的尺度敏感
- 不同特征的量纲差异会影响训练稳定性
- 标准化后训练收敛更快

**学习重点：**
理解特征缩放在神经网络训练中的重要性。

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("First 3 rows before scaling:")
print(X_train[:3])

print("\nFirst 3 rows after scaling:")
print(X_train_scaled[:3])

## 构建一个基线MLP模型

**教学说明（本科生）：**

本节构建一个基础的多层感知机模型。

**模型配置：**
- **隐藏层**：(32, 16) - 两层隐藏层
- **激活函数**：relu
- **优化器**：sgd（随机梯度下降）
- **学习率**：0.01
- **最大迭代次数**：300

**学习重点：**
理解MLPClassifier的基本配置，掌握模型训练方法。

In [ ]:
mlp_baseline = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation="relu",
    solver="sgd",
    learning_rate_init=0.01,
    max_iter=300,
    random_state=42
)

mlp_baseline.fit(X_train_scaled, y_train)

## 基线模型评价

**教学说明（本科生）：**

本节评估基线MLP模型的性能。

**学习重点：**
理解模型评估的基本方法，掌握分类报告的解读。

In [ ]:
y_train_pred = mlp_baseline.predict(X_train_scaled)
y_test_pred = mlp_baseline.predict(X_test_scaled)

print("Baseline MLP")
print("Train accuracy:", accuracy_score(y_train, y_train_pred))
print("Test accuracy:", accuracy_score(y_test, y_test_pred))
print("Classification report on test set:")
print(classification_report(y_test, y_test_pred, target_names=target_names))

## 混淆矩阵

**教学说明（本科生）：**

本节展示分类结果的混淆矩阵。

**学习重点：**
理解混淆矩阵的含义，掌握分类性能评估的详细指标。

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

cm_df = pd.DataFrame(
    cm,
    index=[f"True {name}" for name in target_names],
    columns=[f"Pred {name}" for name in target_names]
)

display(cm_df)

## 查看loss曲线

**教学说明（本科生）：**

本节可视化训练过程中的损失变化。

**学习重点：**
观察损失是否随训练下降，理解训练动态。

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(mlp_baseline.loss_curve_, linewidth=2)
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Baseline MLP loss curve")
plt.show()

## 比较不同的激活函数

**教学说明（本科生）：**

本节比较不同激活函数对模型性能的影响。

**激活函数对比：**
1. **relu**（Rectified Linear Unit）
   - 优点：计算简单，缓解梯度消失
   - 缺点：神经元可能死亡

2. **logistic**（Sigmoid）
   - 优点：输出范围(0,1)
   - 缺点：容易饱和，梯度消失

3. **tanh**（双曲正切）
   - 优点：输出以0为中心
   - 缺点：同样有梯度消失问题

**学习重点：**
理解激活函数对训练效果的影响，掌握如何选择激活函数。

### 如何解读激活函数比较结果

观察输出表格中的 `test_accuracy`：
- **relu**通常表现最好，因为它缓解了梯度消失问题
- **logistic** (sigmoid) 在深层网络中可能收敛较慢
- **tanh** 以0为中心的特性在某些任务上有优势

实际应用中，**ReLU** 是隐藏层的默认首选。

In [ ]:
activation_list = ["relu", "logistic", "tanh"]

activation_results = []

for act in activation_list:

    model = MLPClassifier(

        hidden_layer_sizes=(32, 16),

        activation=act,

        solver="sgd",

        learning_rate_init=0.01,

        max_iter=300,

        random_state=42

    )

    

    model.fit(X_train_scaled, y_train)

    

    y_train_pred_act = model.predict(X_train_scaled)

    y_test_pred_act = model.predict(X_test_scaled)

    

    activation_results.append({

        "activation": act,

        "train_accuracy": accuracy_score(y_train, y_train_pred_act),

        "test_accuracy": accuracy_score(y_test, y_test_pred_act),

        "final_loss": model.loss_curve_[-1],

        "n_iterations": model.n_iter_

    })

activation_results_df = pd.DataFrame(activation_results)

display(activation_results_df)

## 可视化不同**激活函数**的loss曲线

In [ ]:
plt.figure(figsize=(8, 5))

for act in activation_list:
    model = MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation=act,
        solver="sgd",
        learning_rate_init=0.01,
        max_iter=300,
        random_state=42
    )
    model.fit(X_train_scaled, y_train)
    plt.plot(model.loss_curve_, label=act)

plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Loss curves for different activation functions")
plt.legend()
plt.show()

## 比较网络结构

**教学说明（本科生）：**

本节比较不同网络结构对模型性能的影响。

**网络结构对比：**
1. **(10,)**：单层隐藏层，10个神经元
2. **(32, 16)**：两层隐藏层，32和16个神经元
3. **(64, 32, 16)**：三层隐藏层，64、32和16个神经元

**学习重点：**
理解网络深度和宽度对模型性能的影响，体会过拟合和欠拟合。

| 结构 | 参数量 | 表达能力 | 过拟合风险 |
|------|-------|---------|-----------|
| (10,) | 约310 | 低 | 低 |
| (32,16) | 约1500 | 中 | 中 ✅ 通常最佳 |
| (64,32,16) | 约5900 | 高 | 高 |

In [ ]:
structure_list = [
    (10,),
    (32, 16),
    (64, 32, 16)
]

structure_results = []

for structure in structure_list:
    model = MLPClassifier(
        hidden_layer_sizes=structure,
        activation="relu",
        solver="sgd",
        learning_rate_init=0.01,
        max_iter=300,
        random_state=42
    )
    
    model.fit(X_train_scaled, y_train)
    
    y_train_pred_s = model.predict(X_train_scaled)
    y_test_pred_s = model.predict(X_test_scaled)
    
    structure_results.append({
        "hidden_layer_sizes": str(structure),
        "train_accuracy": accuracy_score(y_train, y_train_pred_s),
        "test_accuracy": accuracy_score(y_test, y_test_pred_s),
        "final_loss": model.loss_curve_[-1],
        "n_iterations": model.n_iter_
    })

structure_results_df = pd.DataFrame(structure_results)
display(structure_results_df)

## 比较不同的**梯度下降学习率**

In [ ]:
learning_rates = [0.001, 0.01, 0.1]
lr_results = []

for lr in learning_rates:
    model = MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="sgd",
        learning_rate_init=lr,
        max_iter=300,
        random_state=42
    )
    
    model.fit(X_train_scaled, y_train)
    
    y_train_pred_lr = model.predict(X_train_scaled)
    y_test_pred_lr = model.predict(X_test_scaled)
    
    lr_results.append({
        "learning_rate": lr,
        "train_accuracy": accuracy_score(y_train, y_train_pred_lr),
        "test_accuracy": accuracy_score(y_test, y_test_pred_lr),
        "final_loss": model.loss_curve_[-1],
        "n_iterations": model.n_iter_
    })

lr_results_df = pd.DataFrame(lr_results)
display(lr_results_df)

## 可视化不同学习率的loss曲线

In [ ]:
plt.figure(figsize=(8, 5))

for lr in learning_rates:
    model = MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="sgd",
        learning_rate_init=lr,
        max_iter=300,
        random_state=42
    )
    model.fit(X_train_scaled, y_train)
    plt.plot(model.loss_curve_, label=f"lr={lr}")

plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Loss curves for different learning rates")
plt.legend()
plt.show()

## 打印最终结果

**教学说明（本科生）：**

本节汇总不同超参数配置下的实验结果。

**学习重点：**
理解超参数调优的完整流程，掌握如何从实验中提取有效信息。

In [ ]:
summary_df = pd.DataFrame({
    "Model": [
        "Baseline MLP",
        "Best activation setting candidate",
        "Best structure candidate",
        "Best learning-rate candidate"
    ],
    "Description": [
        "hidden_layer_sizes=(32,16), relu, lr=0.01",
        activation_results_df.loc[activation_results_df["test_accuracy"].idxmax(), "activation"],
        structure_results_df.loc[structure_results_df["test_accuracy"].idxmax(), "hidden_layer_sizes"],
        lr_results_df.loc[lr_results_df["test_accuracy"].idxmax(), "learning_rate"]
    ],
    "Best test accuracy": [
        accuracy_score(y_test, y_test_pred),
        activation_results_df["test_accuracy"].max(),
        structure_results_df["test_accuracy"].max(),
        lr_results_df["test_accuracy"].max()
    ]
})

display(summary_df)

print("Key teaching summary:")
print("1. An artificial neural network can handle real datasets with many features.")
print("2. Activation functions affect the learning behavior of the network.")
print("3. The loss function measures how well the model is fitting the data.")
print("4. MLPClassifier uses gradient-based optimization to reduce the loss.")
print("5. Network structure and learning rate are important hyperparameters.")

## 将所有的参数进行组合，从中筛选出最佳的组合方式

**教学说明（本科生）：**

本节进行网格搜索，找到最优的超参数组合。

**搜索空间：**
- **激活函数**：relu、logistic、tanh
- **网络结构**：(10,)、(32, 16)、(64, 32, 16)
- **学习率**：0.001、0.01、0.1

**组合总数**：3 * 3 * 3 = 27种配置

**学习重点：**
理解超参数组合搜索的方法，掌握如何系统地调优。

## 打印最优结果

In [ ]:
best_row = joint_results_df.iloc[0]

best_activation = best_row["activation"]
best_structure = eval(best_row["hidden_layer_sizes"])
best_lr = best_row["learning_rate"]

print("Best joint combination:")
print("Activation:", best_activation)
print("Hidden layer sizes:", best_structure)
print("Learning rate:", best_lr)
print("Best test accuracy from search:", best_row["test_accuracy"])

## 利用最佳参数进行重新训练

**教学说明（本科生）：**

本节使用找到的最佳超参数重新训练模型。

**完整流程：**
1. 系统搜索所有超参数组合
2. 找到测试准确率最高的配置
3. 使用最佳配置训练最终模型
4. 在测试集上评估最终性能

**学习重点：**
理解超参数调优与模型训练的完整流程。

In [ ]:
mlp_best = MLPClassifier(
    hidden_layer_sizes=best_structure,
    activation=best_activation,
    solver="sgd",
    learning_rate_init=best_lr,
    max_iter=300,
    random_state=42
)

mlp_best.fit(X_train_scaled, y_train)

y_train_best = mlp_best.predict(X_train_scaled)
y_test_best = mlp_best.predict(X_test_scaled)

print("Optimized MLP (joint search)")
print("Best activation:", best_activation)
print("Best structure:", best_structure)
print("Best learning rate:", best_lr)
print("Train accuracy:", accuracy_score(y_train, y_train_best))
print("Test accuracy:", accuracy_score(y_test, y_test_best))

## 可视化loss曲线

**教学说明（本科生）：**

本节可视化最佳模型的损失曲线。

**学习重点：**
观察最佳模型的训练动态，理解损失曲线的含义。

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(mlp_best.loss_curve_, linewidth=2)
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Loss curve of the best jointly optimized MLP")
plt.show()

# 三 神经网络pytorch实现

**教学说明（本科生）：**

本节介绍使用PyTorch实现神经网络。

**PyTorch优势：**
- 动态计算图，灵活性高
- 强大的GPU加速支持
- 自动求导（autograd）
- 广泛的深度学习组件

**学习内容：**
1. PyTorch张量操作
2. 自定义神经网络模型
3. 定义损失函数和优化器
4. 训练循环实现

**学习重点：**
掌握PyTorch的基本使用方法，理解深度学习框架的工作原理。

---
### 学习重点

- 掌握PyTorch的基本张量操作
- 理解nn.Module模型的构建方法
- 掌握PyTorch训练循环的标准流程
- 理解model.train()和model.eval()的区别

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 加载数据集

**教学说明（本科生）：**

本节加载乳腺癌数据集并进行预处理。

**预处理步骤：**
1. 划分训练集和测试集
2. 特征标准化
3. 转换为PyTorch张量

**学习重点：**
理解数据预处理在深度学习中的重要性。

In [ ]:
data = load_breast_cancer()

X = data.data
y = data.target

feature_names = data.feature_names
target_names = data.target_names

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
print("Class names:", target_names)

## 检查数据集

In [ ]:
df = pd.DataFrame(X, columns=feature_names)
df["target"] = y
display(df.head())

## 划分数据集

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 特征缩放

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("First 3 rows after scaling:")
print(X_train_scaled[:3])

## 检查torch张量

**教学说明（本科生）：**

本节将NumPy数组转换为PyTorch张量。

**PyTorch张量：**
- 类似于NumPy数组
- 支持GPU加速
- 自动追踪梯度

**转换方法：**
```
tensor = torch.tensor(numpy_array, dtype=torch.float32)
```

**学习重点：**
理解PyTorch张量的基本操作，掌握数据格式转换。

In [ ]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)

y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32).to(device)

print("X_train_tensor shape:", X_train_tensor.shape)
print("y_train_tensor shape:", y_train_tensor.shape)

## 定义一个两隐藏层前馈神经网络

**教学说明（本科生）：**

本节定义一个简单的前馈神经网络模型。

**网络结构：**
```
输入层（30）→ 隐藏层1（32，ReLU）→ 隐藏层2（16，ReLU）→ 输出层（1）
```

**PyTorch模型定义：**
1. 继承`nn.Module`
2. 定义`__init__`初始化网络层
3. 定义`forward`定义前向传播

**学习重点：**
理解PyTorch模型定义的方法，掌握网络结构的设计。

### PyTorch模型结构

```
输入层(30) → Linear(30→32) → ReLU → Linear(32→16) → ReLU → Linear(16→1) → 输出
```

注意这里最终输出**没有Sigmoid**！因为使用了`BCEWithLogitsLoss`，它内部已经包含了Sigmoid计算，在数值上更稳定。

In [ ]:
class BreastCancerANN(nn.Module):
    def __init__(self, input_dim):
        super(BreastCancerANN, self).__init__()
        
        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    
    def forward(self, x):
        return self.network(x)

## 构建模型

**教学说明（本科生）：**

本节创建模型实例并打印模型结构。

**学习重点：**
理解模型实例化的过程，掌握模型结构的查看方法。

In [ ]:
model = BreastCancerANN(input_dim=X_train.shape[1]).to(device)
print(model)

## 定义损失函数和优化器

**教学说明（本科生）：**

本节定义损失函数和优化器。

**损失函数：**
- **BCEWithLogitsLoss**：二元交叉熵损失（结合sigmoid）
- 适合二分类任务

**优化器：**
- **SGD**：随机梯度下降
- 学习率：0.01

**学习重点：**
理解损失函数和优化器的选择，掌握PyTorch优化器的使用。

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

## 定义精度函数

**教学说明（本科生）：**

本节定义模型精度计算函数。

**精度计算：**
1. 通过sigmoid得到概率
2. 使用阈值0.5进行分类
3. 计算正确预测的比例

**学习重点：**
理解模型评估函数的实现。

In [ ]:
def binary_accuracy_from_logits(logits, y_true):
    probs = torch.sigmoid(logits)
    preds = (probs >= 0.5).float()
    acc = (preds == y_true).float().mean()
    return acc.item()

## Training loop

**教学说明（本科生）：**

本节实现神经网络的训练循环。

**训练步骤：**
1. **前向传播**：计算模型输出
2. **计算损失**：评估预测与真实值的差异
3. **反向传播**：计算梯度
4. **更新参数**：使用优化器更新权重
5. **评估**：计算训练和测试精度

**PyTorch训练模式：**
- `model.train()`：启用训练模式（如dropout）
- `model.eval()`：启用评估模式
- `optimizer.zero_grad()`：清空梯度
- `loss.backward()`：反向传播
- `optimizer.step()`：更新参数

**学习重点：**
掌握PyTorch训练循环的标准流程。

### PyTorch训练循环 标准流程

```python
for epoch in range(num_epochs):
    # 1. 训练模式
    model.train()
    logits = model(X)
    loss = criterion(logits, y)
    
    # 2. 反向传播三件套
    optimizer.zero_grad()  # 清空梯度
    loss.backward()        # 计算梯度
    optimizer.step()       # 更新参数
    
    # 3. 评估模式
    model.eval()
    with torch.no_grad():  # 不计算梯度
        ...
```

> 💡 **教学提示**：`model.train()` 和 `model.eval()` 控制Dropout和BatchNorm等层的行为。在评估时一定要切换到eval模式，否则结果不稳定！

In [ ]:
num_epochs = 300

train_loss_history = []
train_acc_history = []
test_loss_history = []
test_acc_history = []

for epoch in range(num_epochs):
    # ----- training mode -----
    model.train()
    
    logits_train = model(X_train_tensor)
    loss_train = criterion(logits_train, y_train_tensor)
    
    optimizer.zero_grad()
    loss_train.backward()
    optimizer.step()
    
    train_acc = binary_accuracy_from_logits(logits_train, y_train_tensor)
    
    # ----- evaluation mode -----
    model.eval()
    with torch.no_grad():
        logits_test = model(X_test_tensor)
        loss_test = criterion(logits_test, y_test_tensor)
        test_acc = binary_accuracy_from_logits(logits_test, y_test_tensor)
    
    train_loss_history.append(loss_train.item())
    train_acc_history.append(train_acc)
    test_loss_history.append(loss_test.item())
    test_acc_history.append(test_acc)
    
    if epoch % 20 == 0 or epoch == num_epochs - 1:
        print(
            f"Epoch {epoch+1:03d}/{num_epochs} | "
            f"Train Loss = {loss_train.item():.4f} | "
            f"Train Acc = {train_acc:.4f} | "
            f"Test Loss = {loss_test.item():.4f} | "
            f"Test Acc = {test_acc:.4f}"
        )

## 可视化loss曲线

**教学说明（本科生）：**

本节可视化训练和测试损失曲线。

**学习重点：**
观察损失变化，理解模型的训练动态和泛化能力。

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_loss_history, label="Train loss")
plt.plot(test_loss_history, label="Test loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and test loss")
plt.legend()
plt.show()

## 打印最终结果

**教学说明（本科生）：**

本节输出最终模型的测试性能。

**学习重点：**
理解模型评估的完整流程。

In [ ]:
model.eval()

with torch.no_grad():
    logits_test = model(X_test_tensor)
    probs_test = torch.sigmoid(logits_test)
    preds_test = (probs_test >= 0.5).float()

y_test_pred = preds_test.cpu().numpy().astype(int).ravel()

print("Final test accuracy:", accuracy_score(y_test, y_test_pred))

## 分类结果报告

**教学说明（本科生）：**

本节输出分类报告，包括精确率、召回率和F1分数。

**学习重点：**
掌握详细的分类性能评估方法。

In [ ]:
print("Classification report on test set:")
print(classification_report(y_test, y_test_pred, target_names=target_names))

## 混淆矩阵

**教学说明（本科生）：**

本节展示分类结果的混淆矩阵。

**学习重点：**
理解混淆矩阵的含义，掌握分类性能评估的详细指标。

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

cm_df = pd.DataFrame(
    cm,
    index=[f"True {name}" for name in target_names],
    columns=[f"Pred {name}" for name in target_names]
)

display(cm_df)

## 显示一些预测概率

**教学说明（本科生）：**

本节显示一些样本的预测概率。

**学习重点：**
理解模型输出的概率含义，掌握如何解读预测结果。

In [ ]:
results_df = pd.DataFrame({
    "True label": y_test,
    "Predicted label": y_test_pred,
    "Predicted probability": probs_test.cpu().numpy().ravel()
})

display(results_df.head(15))